In [0]:
dbutils.widgets.text("p_data_source", "")

In [0]:
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
%run "../Includes/configs"

In [0]:
%run "../SetUp/setup"

### Access Azure Data Lake using Service Principal
**Steps to follow:**
1. Register Azure AD Application/ Service Principal
2. Generate a secret/ password for the application
3. Set spark config with App/ Client Id, Directory/ Tenant Id & Secret
4. Assign role "Storage Blob Data Contributor" to the Data Lake

path,name,size,modificationTime
abfss://raw@forrmulaa1dl.dfs.core.windows.net/circuits.csv,circuits.csv,10044,1767877118000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/constructors.json,constructors.json,30415,1767877118000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/drivers.json,drivers.json,180812,1767877118000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/lap_times/,lap_times/,0,1767877145000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/pit_stops.json,pit_stops.json,1369387,1767877119000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/qualifying/,qualifying/,0,1767877183000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/races.csv,races.csv,116847,1767877118000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/results.json,results.json,7165641,1767877120000


_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8
circuitId,circuitRef,name,location,country,lat,lng,alt,url
1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,10,http://en.wikipedia.org/wiki/Melbourne_Grand_Prix_Circuit
2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.738,18,http://en.wikipedia.org/wiki/Sepang_International_Circuit
3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,7,http://en.wikipedia.org/wiki/Bahrain_International_Circuit
4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57,2.26111,109,http://en.wikipedia.org/wiki/Circuit_de_Barcelona-Catalunya
5,istanbul,Istanbul Park,Istanbul,Turkey,40.9517,29.405,130,http://en.wikipedia.org/wiki/Istanbul_Park
6,monaco,Circuit de Monaco,Monte-Carlo,Monaco,43.7347,7.42056,7,http://en.wikipedia.org/wiki/Circuit_de_Monaco
7,villeneuve,Circuit Gilles Villeneuve,Montreal,Canada,45.5,-73.5228,13,http://en.wikipedia.org/wiki/Circuit_Gilles_Villeneuve
8,magny_cours,Circuit de Nevers Magny-Cours,Magny Cours,France,46.8642,3.16361,228,http://en.wikipedia.org/wiki/Circuit_de_Nevers_Magny-Cours
9,silverstone,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,http://en.wikipedia.org/wiki/Silverstone_Circuit


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
laptimes_schema = StructType(fields=[StructField("raceId", IntegerType(), False),
                                     StructField("driverId", IntegerType(), True),
                                     StructField("lap", IntegerType(), True),
                                     StructField("position", IntegerType(), True),
                                     StructField("time", StringType(), True),
                                     StructField("milliseconds", IntegerType(), True)])

In [0]:
laptimes_df = spark.read.schema(laptimes_schema).csv(f"{raw_folder_path}/lap_times")

In [0]:
display(laptimes_df)

raceId,driverId,lap,position,time,milliseconds
841,20,1,1,1:38.109,98109
841,20,2,1,1:33.006,93006
841,20,3,1,1:32.713,92713
841,20,4,1,1:32.803,92803
841,20,5,1,1:32.342,92342
841,20,6,1,1:32.605,92605
841,20,7,1,1:32.502,92502
841,20,8,1,1:32.537,92537
841,20,9,1,1:33.240,93240
841,20,10,1,1:32.572,92572


In [0]:
laptimes_df.count()

490904

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
laptimes_final_df = laptimes_df.withColumnRenamed("raceId", "race_id").withColumnRenamed("driverId", "driver_id").withColumn("ingestion_date", current_timestamp()).withColumn("data_source", lit(v_data_source))

In [0]:
laptimes_final_df.write.mode("overwrite").parquet(f"{processed_folder_path}/lap_times")

In [0]:
display(spark.read.parquet(f"{processed_folder_path}/lap_times"))

race_id,driver_id,lap,position,time,milliseconds,ingestion_date,data_source
67,14,26,13,1:25.802,85802,2026-01-10T11:31:42.572518Z,Ergast API
67,14,27,13,1:25.338,85338,2026-01-10T11:31:42.572518Z,Ergast API
67,14,28,13,1:25.395,85395,2026-01-10T11:31:42.572518Z,Ergast API
67,14,29,12,1:26.191,86191,2026-01-10T11:31:42.572518Z,Ergast API
67,14,30,11,1:25.439,85439,2026-01-10T11:31:42.572518Z,Ergast API
67,14,31,10,1:25.375,85375,2026-01-10T11:31:42.572518Z,Ergast API
67,14,32,12,1:28.219,88219,2026-01-10T11:31:42.572518Z,Ergast API
67,14,33,13,1:49.156,109156,2026-01-10T11:31:42.572518Z,Ergast API
67,14,34,13,1:25.128,85128,2026-01-10T11:31:42.572518Z,Ergast API
67,14,35,13,1:25.351,85351,2026-01-10T11:31:42.572518Z,Ergast API
